In [1]:
import pandas as pd
import numpy as np
import glob
import os

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
data_folder = "data"
csv_files = sorted(glob.glob(os.path.join(data_folder, "*.csv")))

if len(csv_files) == 0:
    print("ERROR: No CSV files found. Check the data folder.")
else:
    print(f"Found {len(csv_files)} file(s):")
    for f in csv_files:
        mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {os.path.basename(f):<50}  {mb:.1f} MB")

Found 6 file(s):
  BenignTraffic.pcap.csv                              71.6 MB
  BenignTraffic1.pcap.csv                             58.4 MB
  DDoS-ICMP_Flood.pcap.csv                            47.3 MB
  DDoS-ICMP_Flood15.pcap.csv                          46.5 MB
  DDoS-UDP_Flood.pcap.csv                             47.8 MB
  Recon-PortScan.pcap.csv                             15.7 MB


In [8]:
dfs = []
for f in csv_files:
    fname = os.path.basename(f).lower()
    
    # Determine label from filename
    if 'ddos' in fname or ('dos' in fname and 'recon' not in fname):
        file_label = 'DDoS'
    elif any(x in fname for x in ['recon', 'scan', 'probe', 'sweep']):
        file_label = 'Reconnaissance'
    elif any(x in fname for x in ['benign', 'normal']):
        file_label = 'Benign'
    else:
        print(f"SKIPPING (unrecognised label): {os.path.basename(f)}")
        continue

    temp = pd.read_csv(f, low_memory=False)
    temp['label'] = file_label  # <-- add the label column from the filename
    print(f"Loaded: {os.path.basename(f):<50} {len(temp):>8,} rows  →  {file_label}")
    dfs.append(temp)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nCombined total: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"\nClass distribution:")
print(df_raw['label'].value_counts().to_string())

Loaded: BenignTraffic.pcap.csv                              362,361 rows  →  Benign
Loaded: BenignTraffic1.pcap.csv                             295,585 rows  →  Benign
Loaded: DDoS-ICMP_Flood.pcap.csv                            268,010 rows  →  DDoS
Loaded: DDoS-ICMP_Flood15.pcap.csv                          263,840 rows  →  DDoS
Loaded: DDoS-UDP_Flood.pcap.csv                             266,605 rows  →  DDoS
Loaded: Recon-PortScan.pcap.csv                              82,284 rows  →  Reconnaissance

Combined total: 1,538,685 rows x 40 columns

Class distribution:
label
DDoS              798455
Benign            657946
Reconnaissance     82284


In [9]:
# Find the label column automatically
label_candidates = [c for c in df_raw.columns if 'label' in c.lower()]
print(f"Label column found: {label_candidates}")
label_col = label_candidates[0]

# Show class distribution
print(f"Dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Raw class distribution:")
print(df_raw[label_col].value_counts().to_string())

# Save combined file for notebook 02
df_raw.to_csv("data/combined_raw.csv", index=False)
print(f"Saved combined_raw.csv")

Label column found: ['label']
Dataset shape: 1,538,685 rows x 40 columns
Raw class distribution:
label
DDoS              798455
Benign            657946
Reconnaissance     82284
Saved combined_raw.csv
